# M2.S4 - Introduction to GPU and Accelerator Computing
## Interactive HPC notebook

This notebook accompanies **M2.S4 - Introduction to GPU and Accelerator Computing**.

The goal is not to run Python cells that only reproduce arithmetic from the slides. The goal is to connect the slides to **real measurements on the SciTech GPU**.

### What you will do

1. compare a tiny and a large workload on a real CPU and GPU;
2. see how CUDA block and thread IDs map to actual work;
3. measure the cost of copying data to and from the GPU;
4. run the same vector-add idea with explicit CUDA;
5. run an OpenACC version on the same NVIDIA GPU;
6. connect the programming model to the Slurm resource workflow.

### Classroom method

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

### One GPU allocation, several experiments

To avoid requesting a new GPU for every small activity, this notebook submits **one short Slurm GPU job**. That job runs all the real accelerator examples. Later sections reveal and discuss one part of the output at a time.

> **Concepts can be explained with text. Performance claims should be measured.**

> **Notebook build: M2S4-2026-09-22-v7**
>
> This version replaces synthetic Python calculations with one real GPU mini-lab using the validated SciTech NVIDIA HPC SDK route: `nvhpc/25.7`.

# 0 - Check the environment

### Predict

Before running:

- Is this Jupyter kernel itself a GPU allocation?
- Which scheduler controls access to the shared GPU?
- What is the difference between seeing NVIDIA software and actually owning a GPU?

In [ ]:
import os
import platform
import re
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

print("Host:", platform.node())
print("User:", os.environ.get("USER", "unknown"))
print("Current Slurm job:", os.environ.get("SLURM_JOB_ID", "not set"))
print("Current Slurm partition:", os.environ.get("SLURM_JOB_PARTITION", "not set"))
print("sbatch:", shutil.which("sbatch"))
print("sinfo:", shutil.which("sinfo"))
print("nvidia-smi visible in PATH:", shutil.which("nvidia-smi"))

### Explain

Seeing `nvidia-smi`, CUDA libraries or NVIDIA tools on a system does **not** mean your current process owns a GPU.

On this cluster:

```text
Jupyter session
      |
      | sbatch requests a GPU
      v
Slurm GPU partition
      |
      v
GPU compute node
```

Slurm controls access to the accelerator.

# 1 - Prepare the real examples

The examples are stored with the course repository under:

```text
session_demos/09_gpu/
```

The next cell copies them from your local course repository if it exists. Otherwise it downloads the same files from GitHub.

You do not need to study this setup cell. Its purpose is only to place the compiled examples beside the notebook.

In [ ]:
DEMO_NAMES = [
    "quick_compare.cu",
    "thread_mapping.cu",
    "data_movement.cu",
    "vector_add.cu",
    "openacc_vector_add.c",
    "gpu_lab.sbatch",
]

RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "OscarDiez/hpc_course/main/session_demos/09_gpu/"
)

LOCAL_DEMO = Path.home() / "hpc_course" / "session_demos" / "09_gpu"

for name in DEMO_NAMES:
    target = Path(name)

    if (LOCAL_DEMO / name).exists():
        shutil.copy2(LOCAL_DEMO / name, target)
        source = "local ~/hpc_course"
    else:
        urllib.request.urlretrieve(RAW_BASE + name, target)
        source = "GitHub"

    print(f"{name:<26} <- {source}")

print("\nExamples ready.")

# 2 - Submit one real GPU mini-lab

The job requests:

- the `gpu` partition;
- one GPU;
- two CPU resources;
- 4 GB of RAM;
- at most five minutes.

Inside the allocated node it loads the validated:

```bash
module load nvhpc/25.7
```

and runs five short experiments.

### Predict

Before submitting:

1. Which GPU model do you expect?
2. Will the tiny workload favor the CPU or GPU end-to-end?
3. Will keeping data on the GPU help repeated operations?

In [ ]:
def clean_slurm_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit_slurm(script):
    p = subprocess.run(
        ["sbatch", "--parsable", script],
        capture_output=True,
        text=True,
        env=clean_slurm_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr.strip() or "sbatch failed")
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def wait_for_job(job_id, timeout=300, poll=3):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True,
            text=True
        )
        state = p.stdout.strip()
        if not state:
            print("Job", job_id, "has left the queue.")
            return True
        print("Job", job_id, "state:", state)
        time.sleep(poll)
    print("Timed out. The job may still be queued or running.")
    return False

def read_job_output(job_id):
    path = Path(f"m2s4_lab-{job_id}.out")
    if not path.exists():
        print("Output file not found yet:", path)
        return ""
    return path.read_text(errors="replace")

def show_section(text, number):
    start = f"===== EXPERIMENT {number}:"
    end = f"===== END EXPERIMENT {number} ====="

    i = text.find(start)
    j = text.find(end)

    if i < 0 or j < 0:
        print("Experiment section not found.")
        return

    print(text[i:j + len(end)])

print("Slurm helpers ready.")

In [ ]:
if shutil.which("sinfo"):
    print("--- GPU partition ---")
    subprocess.run(
        ["sinfo", "-p", "gpu", "-o", "%P %a %l %D %c %G"],
        check=False
    )

M2S4_JOB_ID = submit_slurm("gpu_lab.sbatch")

if wait_for_job(M2S4_JOB_ID, timeout=300):
    time.sleep(1)
    M2S4_LAB_OUTPUT = read_job_output(M2S4_JOB_ID)
    print("\nGPU mini-lab completed.")
else:
    M2S4_LAB_OUTPUT = ""

# 3 - Real example 1: when does the GPU win?
### CPU latency vs GPU throughput

The experiment performs the **same vector addition** on CPU and GPU.

It measures two cases:

- **tiny**: 1,024 elements, one operation;
- **large + reuse**: 5,000,000 elements, repeated 20 times.

For the GPU it reports:

- **GPU kernel time**: data is already on the GPU;
- **GPU total time**: input copies + kernels + result copy.

### Predict

Which timing is the fair comparison with CPU time if the application must move its data to the GPU?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 1)

In [ ]:
def parse_case(text, label):
    pattern = (
        rf"^{re.escape(label)}\s+(\d+)\s+(\d+)\s+"
        rf"([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+(PASS|FAIL)$"
    )
    m = re.search(pattern, text, re.MULTILINE)
    if not m:
        return None
    return {
        "N": int(m.group(1)),
        "reps": int(m.group(2)),
        "cpu_ms": float(m.group(3)),
        "kernel_ms": float(m.group(4)),
        "gpu_total_ms": float(m.group(5)),
        "check": m.group(6),
    }

tiny = parse_case(M2S4_LAB_OUTPUT, "tiny")
large = parse_case(M2S4_LAB_OUTPUT, "large_reuse")

for name, result in [("tiny", tiny), ("large + reuse", large)]:
    if result:
        speedup = result["cpu_ms"] / result["gpu_total_ms"]
        print(
            f"{name:<14} end-to-end speedup = {speedup:.2f}x "
            f"(CPU {result['cpu_ms']:.4f} ms / GPU total {result['gpu_total_ms']:.4f} ms)"
        )

### Explain

The important comparison is:

```text
CPU time  vs  GPU total time
```

Kernel-only time answers a different question:

> How fast is the accelerator computation **after the data is already there**?

For very small work, launch and transfer overhead can cost more than the computation itself.

For large, regular work with reuse, the GPU has enough parallel work to make its high throughput useful.

# 4 - Real example 2: CUDA blocks and threads

The slides introduce:

```c
int i = blockIdx.x * blockDim.x + threadIdx.x;
```

Instead of calculating a few indices in Python, the real CUDA kernel records its own block and thread IDs.

This experiment uses:

- 20 useful elements;
- 8 threads per block;
- 3 blocks;
- 24 launched threads.

### Predict

- Which global index is `block 1, thread 3`?
- How many of the 24 launched threads are extra?
- Why does normal vector-add code need `if (i < n)`?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 2)

### Explain

For one-dimensional CUDA indexing:

```text
global index = block number x threads per block + thread number
```

So:

```text
block 1, thread 3
= 1 x 8 + 3
= global index 11
```

The last block is only partly useful. CUDA launches complete blocks, so extra threads must be prevented from accessing memory beyond the end of the array.

That is why the vector-add kernel uses:

```c
if (i < n)
    c[i] = a[i] + b[i];
```

# 5 - Real example 3: the cost of moving data

Now the GPU performs the same 2,000,000-element vector operation 20 times in two ways.

**Strategy A - copy every iteration**

```text
CPU -> GPU
compute
GPU -> CPU
repeat 20 times
```

**Strategy B - keep data resident**

```text
CPU -> GPU once
compute x 20
GPU -> CPU once
```

### Predict

Which strategy should be faster, and why?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 3)

In [ ]:
copy_m = re.search(r"copy_every_iteration_ms=([0-9.]+)", M2S4_LAB_OUTPUT)
resident_m = re.search(r"keep_data_resident_ms=([0-9.]+)", M2S4_LAB_OUTPUT)

if copy_m and resident_m:
    copy_ms = float(copy_m.group(1))
    resident_ms = float(resident_m.group(1))
    print(f"Copy every iteration : {copy_ms:.3f} ms")
    print(f"Keep data resident   : {resident_ms:.3f} ms")
    print(f"Benefit from reuse   : {copy_ms/resident_ms:.2f}x")
else:
    print("Timing values not found.")

### Explain

This is the practical version of:

```text
GPU time = copy in + compute + copy out
```

A fast GPU kernel does not guarantee a fast application.

If the same data is used by many GPU operations, keeping it on the device can remove repeated transfer cost.

> **Move data less often. Do more useful work while it is on the GPU.**

# 6 - Real example 4: minimal CUDA vector addition

The important CUDA code is small:

```c
__global__ void vector_add(const float *a,
                           const float *b,
                           float *c,
                           int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
        c[i] = a[i] + b[i];
}
```

The launch is:

```c
vector_add<<<blocks, 256>>>(d_a, d_b, d_c, n);
```

For 1,024 elements:

```text
1024 elements / 256 threads per block = 4 blocks
```

### Predict

How many useful elements does each CUDA thread handle in this example?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 4)

### Explain

Each useful CUDA thread handles **one vector element**.

CUDA gives the programmer explicit control over:

- GPU memory allocation;
- CPU-to-GPU and GPU-to-CPU copies;
- block size;
- number of blocks;
- kernel launch.

That control is powerful, but it also means more code and more responsibility.

# 7 - Real example 5: the same idea with OpenACC

OpenACC lets the compiler generate accelerator code from directives.

The central part of the example is:

```c
#pragma acc data copyin(a[0:n], b[0:n]) copyout(c[0:n])
{
    for (int r = 0; r < reps; ++r) {

        #pragma acc parallel loop present(a[0:n], b[0:n], c[0:n])
        for (int i = 0; i < n; ++i)
            c[i] = a[i] + b[i] + 0.0001f*r;
    }
}
```

Compare that with CUDA.

**CUDA**
- explicit device allocation;
- explicit copies;
- explicit kernel;
- explicit blocks and threads.

**OpenACC**
- directives describe what should run on the accelerator;
- the compiler generates the GPU kernel;
- the data region keeps arrays on the GPU across repeated loops.

### Predict

Which approach gives more low-level control? Which requires fewer source changes?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 5)

### Explain

- **CUDA** gives more explicit control over execution and memory.
- **OpenACC** usually requires fewer changes to existing scientific C or Fortran code.
- Both examples above execute on the real NVIDIA GPU.
- Neither is automatically faster in every application. Measure the complete workload.

A useful rule is:

> Use the highest-level solution that gives you the control and performance you actually need.

# 8 - Two ways to reach the same SciTech GPU

## From this JupyterHub notebook

Jupyter is already inside a Slurm-managed allocation, so use:

```text
JupyterHub -> sbatch -> GPU partition -> GPU compute node
```

That is the workflow used in this notebook.

## From the normal SSH login node

A short interactive demonstration can use:

```bash
srun -p gpu --gpus=1 --cpus-per-task=2 --mem=4G --time=00:10:00 --pty bash -l
```

Then:

```bash
hostname
nvidia-smi
module purge
module load nvhpc/25.7
```

Do **not** start that interactive `srun --pty` route from inside the existing Jupyter Slurm allocation.

The CUDA programming model is the same. Only the way you request the resource changes.

# 9 - Which programming approach would you choose?

Choose a sensible first approach.

### A
A 10,000-line scientific C application has one loop that consumes 70% of runtime.

### B
A new NVIDIA-specific algorithm needs explicit control of threads, memory and execution.

### C
The hot operation is standard dense matrix multiplication.

<details>
<summary><strong>Show suggested answer</strong></summary>

- **A - OpenACC** is a plausible first approach because it can accelerate existing loop-based code with relatively small source changes.
- **B - CUDA** fits when NVIDIA-specific low-level control is important.
- **C - optimized GPU library** should usually be tried first when a high-quality implementation already exists.

</details>

# 10 - Challenge: diagnose a GPU result

A team reports:

```text
CPU application       = 200 ms
GPU kernel            =  20 ms
complete GPU program  = 150 ms
```

Answer:

1. Is the GPU kernel 10x faster than the CPU computation?
2. Is the application 10x faster?
3. What is probably consuming the missing time?
4. What could help if the same data is used by 50 GPU kernels?

<details>
<summary><strong>Show suggested solution</strong></summary>

1. Yes. At kernel level, 200 / 20 = 10x.
2. No. End-to-end speedup is only 200 / 150 = 1.33x.
3. Investigate transfers, allocation, synchronization and launch overhead.
4. Keep the data resident on the GPU and perform many kernels before copying the final result back.

</details>

# What did we learn?

1. **A GPU is not automatically faster.** Small work can favor the CPU.
2. **GPUs are good at throughput.** Large, regular, independent work is a natural fit.
3. **CUDA maps work through grids, blocks and threads.**
4. **Extra threads are normal.** Boundary checks keep them safe.
5. **Data movement matters.** Repeated copies can dominate runtime.
6. **Keeping data resident can change the result dramatically.**
7. **CUDA and OpenACC can run the same idea at different abstraction levels.**
8. **Measure end-to-end performance, not only kernel time.**
9. **Use Slurm when you actually need the shared GPU.**

### Source examples

The programs used by this notebook are also available here:

```text
https://github.com/OscarDiez/hpc_course/tree/main/session_demos/09_gpu
```

The next session continues the same HPC question:

> **Where is the real bottleneck, and what programming model matches the architecture?**